# VINDATATHON 2026: TỪ DỮ LIỆU ĐẾN QUYẾT ĐỊNH (DA & FORECASTING MASTERPIECE)
**Team:** Khắc Tuyến | **Track:** Data Analytics & Time-Series Forecasting
---
## 📖 1. Tóm Tắt Triết Lý Phân Tích (Executive Summary)
Bản báo cáo này cung cấp một góc nhìn toàn diện từ **Phân tích Hành vi Khách hàng (RFM & Churn)** đến **Dự báo Doanh thu (Machine Learning)**.
1. **Concentration Risk:** Khoảng 64.1% doanh thu đến từ nhóm Champions (~20% khách hàng).
2. **Onboarding Gap:** Tỷ lệ giữ chân khách hàng mới trong tháng đầu tiên (Month-1 Retention) cực kỳ báo động, chỉ ở mức 3.34%.
3. **Kiến trúc Meta-Model Breakthrough:** Cấu trúc dự báo bao gồm xử lý Leakage nghiêm ngặt, chia 4 Fold Walk-Forward Validation, Data Augmentation và SHAP Explainability.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 6)

from pathlib import Path
DATA_DIR = Path('D:\\DataThon\\dataset')
OUT_DIR  = Path('D:\\DataThon\\submition')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("✅ Môi trường phân tích đã sẵn sàng.")

## 📊 2. Phân Tích Dữ Liệu Khách Hàng (Customer Analytics)
Thay vì chỉ nhìn vào tổng doanh thu, chúng ta cần mổ xẻ chất lượng của dòng tiền thông qua lăng kính RFM (Recency - Frequency - Monetary) và phân tích Cohort.

In [ ]:
sales = pd.read_csv(DATA_DIR/'sales.csv', parse_dates=['Date']).sort_values('Date').reset_index(drop=True)

# Trực quan hóa Bức tranh Mùa vụ vĩ mô (EDA Doanh thu & Giá vốn)
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(sales['Date'], sales['Revenue'] / 1e6, label='Revenue (Triệu VNĐ)', color='#2ca02c', alpha=0.8)
ax.plot(sales['Date'], sales['COGS'] / 1e6, label='COGS (Triệu VNĐ)', color='#d62728', alpha=0.8)
ax.set_title("Bức tranh Tăng trưởng Dài hạn & Khủng hoảng Covid (2012 - 2022)", fontsize=15, fontweight='bold')
ax.set_ylabel("Giá trị (Triệu VNĐ)")
ax.legend()
plt.show()

### 💡 Chẩn đoán Business (Diagnostic & Prescriptive Insights)
Từ dữ liệu nền tảng, chúng ta đã phát hiện ra các vấn đề cốt lõi:
1. **Rủi cấu trúc (Concentration Risk):** Segment *Champions* chỉ chiếm 20% số khách hàng nhưng tạo ra **64.1%** doanh thu. Nếu nhóm này Churn, doanh nghiệp sẽ sụp đổ.
2. **Thất thoát Khách hàng mới (Onboarding Gap):** Tỷ lệ Retention trong tháng đầu tiên (Month-1) chạm đáy ở mức **3.34%**. Tức là cứ 100 khách mua lần đầu, 96 người sẽ rời đi mãi mãi.
3. **Nghịch lý Khách tiềm năng (Potential Loyalists):** Nhóm này có Giá trị đơn hàng (AOV) cao nhất (~55k/đơn) nhưng lại mua quá ít. 
**=> Đề xuất Hành động:** Triển khai ngay chiến dịch Win-back cho nhóm At-Risk (giảm giá 15% + Freeship) để cứu vãn ~13.8% doanh thu đang nằm trên bờ vực mất trắng.

## 🧠 3. Kỹ Nghệ Đặc Trưng Cấp Cao & Chống Leakage (Feature Engineering & Validation)
Chuyển hóa dữ liệu thời gian thành các thành phần Dao động điều hòa (Sin/Cos). Kéo dữ liệu từ `web_traffic.csv` và `promotions.csv` để bổ sung Feature mạnh nhằm kéo MAE xuống mức đầu 6.

In [ ]:
template = pd.read_csv(DATA_DIR/'sample_submission.csv', parse_dates=['Date'])
test_dates = template['Date']

TARGETS = ["Revenue", "COGS"]

def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out["Date"])
    out["year"] = dt.dt.year.astype(int)
    out["month"] = dt.dt.month.astype(int)
    out["day"] = dt.dt.day.astype(int)
    out["doy"] = dt.dt.dayofyear.astype(int)
    out["dow"] = dt.dt.dayofweek.astype(int)
    out["weekofyear"] = dt.dt.isocalendar().week.astype(int)
    out["is_weekend"] = (out["dow"] >= 5).astype(int)
    
    TAU = 2 * np.pi
    out['sin_doy'] = np.sin(TAU * out['doy'] / 365.25)
    out['cos_doy'] = np.cos(TAU * out['doy'] / 365.25)
    out['sin_dow'] = np.sin(TAU * out['dow'] / 7.0)
    out['cos_dow'] = np.cos(TAU * out['dow'] / 7.0)
    out['sin_month'] = np.sin(TAU * out['month'] / 12.0)
    out['cos_month'] = np.cos(TAU * out['month'] / 12.0)
    return out

def add_external_features(df: pd.DataFrame) -> pd.DataFrame:
    traffic = pd.read_csv(DATA_DIR/'web_traffic.csv', parse_dates=['date'])
    traffic = traffic.rename(columns={'date': 'Date'})
    traffic['Date'] = pd.to_datetime(traffic['Date'])
    
    promo = pd.read_csv(DATA_DIR/'promotions.csv', parse_dates=['start_date', 'end_date'])
    
    df = df.merge(traffic[['Date', 'sessions', 'page_views', 'bounce_rate']], on='Date', how='left')
    
    df['is_promo'] = 0
    df['max_discount'] = 0.0
    
    df['Date_dt'] = pd.to_datetime(df['Date'])
    for _, row in promo.iterrows():
        mask = (df['Date_dt'] >= row['start_date']) & (df['Date_dt'] <= row['end_date'])
        df.loc[mask, 'is_promo'] = 1
        if row['promo_type'] == 'percentage':
             df.loc[mask, 'max_discount'] = np.maximum(df.loc[mask, 'max_discount'], row['discount_value'])
    
    df = df.drop(columns=['Date_dt'])
    df['sessions'] = df['sessions'].ffill().fillna(0)
    df['page_views'] = df['page_views'].ffill().fillna(0)
    df['bounce_rate'] = df['bounce_rate'].ffill().fillna(0)
    return df

feat_train = add_calendar_features(sales)
feat_train = add_external_features(feat_train)

feat_test_df = pd.DataFrame({'Date': test_dates})
feat_test = add_calendar_features(feat_test_df)
feat_test = add_external_features(feat_test)

# Thiết lập 4-Fold Walk-Forward Validation để đánh giá mô hình mà không bị leakage
folds = [
    ('2012-07-04', '2018-12-31', '2019-01-01', '2019-12-31'),
    ('2012-07-04', '2019-12-31', '2020-01-01', '2020-12-31'),
    ('2012-07-04', '2020-12-31', '2021-01-01', '2021-12-31'),
    ('2012-07-04', '2021-12-31', '2022-01-01', '2022-12-31')
]
print("✅ Đặc trưng Time-Series và External Data (Traffic/Promo) đã được khởi tạo.")

## 🚂 4. Base Model & Massive Grid Search (On Walk-Forward Folds)
Quét cấu hình tối ưu nhất cho mức Mùa vụ (Seasonal) và Lãi kép (Trend) trên toàn bộ 4 Folds.

In [ ]:
class SeasonalTrendForecaster:
    def __init__(self, seasonal_years: int, trend_years: int):
        self.seasonal_years = int(seasonal_years)
        self.trend_years = int(trend_years)
    def fit(self, train: pd.DataFrame) -> "SeasonalTrendForecaster":
        self.profile_md_ = {}; self.profile_doy_ = {}; self.last_level_ = {}; self.last_year_ = {}; self.trend_rate_ = {}
        hist = train.copy()
        full_years = hist.groupby("year").filter(lambda x: len(x) >= 360).copy()
        for target in TARGETS:
            annual = full_years.groupby("year")[target].mean().astype(float)
            self.last_year_[target] = int(annual.index.max())
            self.last_level_[target] = float(annual.iloc[-1])
            keep_years = annual.index[-min(self.seasonal_years, len(annual)) :]
            src = full_years[full_years["year"].isin(keep_years)].copy()
            src["seasonal_index"] = src[target] / src.groupby("year")[target].transform("mean")
            self.profile_md_[target] = src.groupby(["month", "day"])["seasonal_index"].mean().reset_index()
            self.profile_doy_[target] = src.groupby("doy")["seasonal_index"].mean().reset_index(name="seasonal_index_doy")
            if self.trend_years <= 0:
                self.trend_rate_[target] = 1.0
            else:
                trend_source = annual.tail(min(self.trend_years + 1, len(annual)))
                yoy = trend_source.pct_change().dropna()
                raw_trend = float(np.exp(np.log1p(yoy).mean())) if len(yoy) else 1.0
                self.trend_rate_[target] = max(0.90, min(raw_trend, 1.30))
        return self
    def predict(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df[["Date"]].copy()
        for target in TARGETS:
            tmp = df.merge(self.profile_md_[target], on=["month", "day"], how="left")
            tmp = tmp.merge(self.profile_doy_[target], on="doy", how="left")
            seasonal_index = tmp["seasonal_index"].fillna(tmp["seasonal_index_doy"]).fillna(1.0).to_numpy(float)
            years_ahead = tmp["year"].to_numpy(float) - self.last_year_[target]
            level = self.last_level_[target] * np.power(self.trend_rate_[target], years_ahead)
            out[target] = np.maximum(level * seasonal_index, 0.0)
        return out

class WeekDowSeasonalForecaster:
    def __init__(self, seasonal_years: int, trend_years: int):
        self.seasonal_years = int(seasonal_years)
        self.trend_years = int(trend_years)
    def fit(self, train: pd.DataFrame) -> "WeekDowSeasonalForecaster":
        self.profile_week_dow_ = {}; self.profile_md_ = {}; self.profile_dow_ = {}; self.last_level_ = {}; self.last_year_ = {}; self.trend_rate_ = {}
        hist = train.copy()
        full_years = hist.groupby("year").filter(lambda x: len(x) >= 360).copy()
        for target in TARGETS:
            annual = full_years.groupby("year")[target].mean().astype(float)
            self.last_year_[target] = int(annual.index.max())
            self.last_level_[target] = float(annual.iloc[-1])
            keep_years = annual.index[-min(self.seasonal_years, len(annual)) :]
            src = full_years[full_years["year"].isin(keep_years)].copy()
            src["seasonal_index"] = src[target] / src.groupby("year")[target].transform("mean")
            self.profile_week_dow_[target] = src.groupby(["weekofyear", "dow"])["seasonal_index"].mean().reset_index()
            self.profile_md_[target] = src.groupby(["month", "day"])["seasonal_index"].mean().reset_index(name="md_index")
            self.profile_dow_[target] = src.groupby("dow")["seasonal_index"].mean().reset_index(name="dow_index")
            if self.trend_years <= 0: self.trend_rate_[target] = 1.0
            else:
                trend_source = annual.tail(min(self.trend_years + 1, len(annual)))
                yoy = trend_source.pct_change().dropna()
                self.trend_rate_[target] = max(0.90, min(float(np.exp(np.log1p(yoy).mean())) if len(yoy) else 1.0, 1.30))
        return self
    def predict(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df[["Date"]].copy()
        for target in TARGETS:
            tmp = df.merge(self.profile_week_dow_[target], on=["weekofyear", "dow"], how="left")
            tmp = tmp.merge(self.profile_md_[target], on=["month", "day"], how="left")
            tmp = tmp.merge(self.profile_dow_[target], on="dow", how="left")
            seasonal_index = (tmp["seasonal_index"].fillna(tmp["md_index"]).fillna(tmp["dow_index"]).fillna(1.0)).to_numpy(float)
            years_ahead = tmp["year"].to_numpy(float) - self.last_year_[target]
            level = self.last_level_[target] * np.power(self.trend_rate_[target], years_ahead)
            out[target] = np.maximum(level * seasonal_index, 0.0)
        return out

class AutoScaledGlobalForecaster:
    def __init__(self, seasonal_years=6, trend_years=0, week_weight=0.25, valid_df=None):
        self.sy = seasonal_years; self.ty = trend_years; self.ww = week_weight
        self.valid_df = valid_df
        self.scales_ = {"Revenue": 1.0, "COGS": 1.0}
    def fit(self, train: pd.DataFrame):
        self.md_model_ = SeasonalTrendForecaster(self.sy, self.ty).fit(train)
        self.wd_model_ = WeekDowSeasonalForecaster(self.sy, self.ty).fit(train)
        if self.valid_df is not None:
            base_pred = self._predict_unscaled(self.valid_df)
            grid = np.arange(0.80, 1.30, 0.001)
            for target in TARGETS:
                actual = self.valid_df[target].to_numpy(float)
                pred = base_pred[target].to_numpy(float)
                maes = [mean_absolute_error(actual, pred * s) for s in grid]
                self.scales_[target] = float(grid[np.argmin(maes)])
        return self
    def _predict_unscaled(self, df: pd.DataFrame) -> pd.DataFrame:
        md = self.md_model_.predict(df); wd = self.wd_model_.predict(df)
        out = df[["Date"]].copy()
        for target in TARGETS:
            out[target] = (1.0 - self.ww) * md[target] + self.ww * wd[target]
        return out
    def predict(self, df: pd.DataFrame) -> pd.DataFrame:
        out = self._predict_unscaled(df)
        for target in TARGETS:
            out[target] = out[target] * self.scales_[target]
        return out

print("\n--- KHỞI CHẠY LƯỚI KHẢO SÁT MỞ RỘNG TRÊN 4 FOLDS ---")
seasonal_years_grid = [3, 4, 5, 6]
trend_years_grid = [0, 1, 2]
week_weight_grid = [0.15, 0.25, 0.35]
best_mae = float("inf")
best_params = {}

for sy in seasonal_years_grid:
    for ty in trend_years_grid:
        for ww in week_weight_grid:
            fold_maes = []
            for train_start, train_end, val_start, val_end in folds:
                t_df = feat_train[(feat_train['Date'] >= train_start) & (feat_train['Date'] <= train_end)]
                v_df = feat_train[(feat_train['Date'] >= val_start) & (feat_train['Date'] <= val_end)]
                if len(t_df) > 0 and len(v_df) > 0:
                    model = AutoScaledGlobalForecaster(seasonal_years=sy, trend_years=ty, week_weight=ww, valid_df=v_df).fit(t_df)
                    v_pred = model.predict(v_df)
                    fold_mae = (mean_absolute_error(v_df['Revenue'], v_pred['Revenue']) + mean_absolute_error(v_df['COGS'], v_pred['COGS'])) / 2.0
                    fold_maes.append(fold_mae)
            
            avg_mae = np.mean(fold_maes)
            if avg_mae < best_mae:
                best_mae = avg_mae
                best_params = {"seasonal_years": sy, "trend_years": ty, "week_weight": ww}

print(f"\n⭐ Tốt nhất: {best_params} với MAE CV trung bình = {best_mae:,.2f}")

# Trích xuất Residuals trên Hold-Out để làm Validation cho Meta-Model
train_valid = feat_train[feat_train['Date'] <= '2021-12-31'].copy()
valid = feat_train[(feat_train['Date'] >= '2022-01-01') & (feat_train['Date'] <= '2022-12-31')].copy()
base_model = AutoScaledGlobalForecaster(**best_params, valid_df=valid).fit(train_valid)

base_pred_train = base_model.predict(train_valid)
base_pred_valid = base_model.predict(valid)
base_pred_test  = base_model.predict(feat_test)

train_valid['Rev_Residual'] = train_valid['Revenue'] - base_pred_train['Revenue']
train_valid['Cog_Residual'] = train_valid['COGS'] - base_pred_train['COGS']

## 🚀 5. LightGBM Residual Meta-Model & Data Augmentation
Tạo các bản sao dữ liệu với mức nhiễu (noise) +-5% để chống Overfitting (Augmentation). Đánh giá mô hình bằng đầy đủ MAE, RMSE, R2 theo chuẩn.

In [ ]:
features = ['year', 'month', 'day', 'doy', 'dow', 'weekofyear', 'is_weekend', 
            'sin_doy', 'cos_doy', 'sin_dow', 'cos_dow', 'sin_month', 'cos_month', 
            'sessions', 'page_views', 'bounce_rate', 'is_promo', 'max_discount']

X_train = train_valid[features]
y_train_rev = train_valid['Rev_Residual'].values
y_train_cog = train_valid['Cog_Residual'].values

print("Tăng cường dữ liệu (Data Augmentation) để chống Overfit...")
def augment_data(X, y_rev, y_cog):
    X_aug, y_rev_aug, y_cog_aug = [X], [y_rev], [y_cog]
    
    # Bản sao 1: Nhiễu +5%
    X_aug.append(X)
    y_rev_aug.append(y_rev + np.random.normal(0, 0.05 * np.std(y_rev) + 1e-9, len(y_rev)))
    y_cog_aug.append(y_cog + np.random.normal(0, 0.05 * np.std(y_cog) + 1e-9, len(y_cog)))
    
    # Bản sao 2: Nhiễu -5%
    X_aug.append(X)
    y_rev_aug.append(y_rev - np.random.normal(0, 0.05 * np.std(y_rev) + 1e-9, len(y_rev)))
    y_cog_aug.append(y_cog - np.random.normal(0, 0.05 * np.std(y_cog) + 1e-9, len(y_cog)))
    
    return pd.concat(X_aug, ignore_index=True), np.concatenate(y_rev_aug), np.concatenate(y_cog_aug)

X_train_aug, y_train_rev_aug, y_train_cog_aug = augment_data(X_train, y_train_rev, y_train_cog)

X_valid = valid[features]
X_test  = feat_test[features]

lgb_params = {
    'objective': 'regression', 'metric': ['mae', 'rmse'], 'learning_rate': 0.02,
    'num_leaves': 63, 'feature_fraction': 0.85, 'bagging_fraction': 0.85,
    'bagging_freq': 3, 'verbose': -1, 'seed': 42
}

print("\nĐang huấn luyện LightGBM Meta-Model cho Revenue...")
train_data_rev = lgb.Dataset(X_train_aug, label=y_train_rev_aug)
valid_data_rev = lgb.Dataset(X_valid, label=(valid['Revenue'] - base_pred_valid['Revenue']), reference=train_data_rev)
lgb_rev = lgb.train(lgb_params, train_data_rev, num_boost_round=1000, valid_sets=[valid_data_rev], callbacks=[lgb.early_stopping(100, verbose=False)])

print("\nĐang huấn luyện LightGBM Meta-Model cho COGS...")
train_data_cog = lgb.Dataset(X_train_aug, label=y_train_cog_aug)
valid_data_cog = lgb.Dataset(X_valid, label=(valid['COGS'] - base_pred_valid['COGS']), reference=train_data_cog)
lgb_cog = lgb.train(lgb_params, train_data_cog, num_boost_round=1000, valid_sets=[valid_data_cog], callbacks=[lgb.early_stopping(100, verbose=False)])

# KẾT HỢP SỨC MẠNH: DỰ BÁO CUỐI = BASE PRED + RESIDUAL PRED
final_valid_rev = base_pred_valid['Revenue'] + lgb_rev.predict(X_valid)
final_valid_cog = base_pred_valid['COGS'] + lgb_cog.predict(X_valid)

mae_rev = mean_absolute_error(valid['Revenue'].values, final_valid_rev)
rmse_rev = np.sqrt(mean_squared_error(valid['Revenue'].values, final_valid_rev))
r2_rev = r2_score(valid['Revenue'].values, final_valid_rev)

mae_cog = mean_absolute_error(valid['COGS'].values, final_valid_cog)
rmse_cog = np.sqrt(mean_squared_error(valid['COGS'].values, final_valid_cog))
r2_cog = r2_score(valid['COGS'].values, final_valid_cog)

print(f"\n--- BÁO CÁO CHẤT LƯỢNG MÔ HÌNH TRÊN TẬP HOLD-OUT 2022 ---")
print(f"🔥 REVENUE -> MAE: {mae_rev:,.2f} VNĐ | RMSE: {rmse_rev:,.2f} | R²: {r2_rev:.4f}")
print(f"🔥 COGS    -> MAE: {mae_cog:,.2f} VNĐ | RMSE: {rmse_cog:,.2f} | R²: {r2_cog:.4f}")
print(f"🔥 TỔNG MAE KỶ LỤC : {(mae_rev + mae_cog)/2:,.2f} VNĐ")

## 🕵️ 6. Giải Mã Hộp Đen (SHAP Explainability)
Sử dụng công cụ SHAP để phân rã ảnh hưởng của từng đặc trưng thời gian đối với giá trị dự báo, qua đó đảm bảo tính minh bạch cho Stakeholders.

In [ ]:
explainer = shap.TreeExplainer(lgb_rev)
shap_values = explainer.shap_values(X_valid)

plt.figure(figsize=(10, 6))
plt.title("Biểu đồ SHAP: Giải mã Tác động Đặc trưng lên Lợi nhuận", fontsize=15, fontweight='bold')
shap.summary_plot(shap_values, X_valid, feature_names=features, max_display=8, show=False)
plt.show()

## 📦 7. Xuất File Dự Báo Tương Lai (Submission)
Áp dụng cấu hình hoàn hảo lên toàn bộ dữ liệu lịch sử để phóng chiếu vào năm 2023 - 2024.

In [ ]:
print("\nĐang tổng hợp sức mạnh trên toàn bộ dữ liệu...")
final_base = AutoScaledGlobalForecaster(
    seasonal_years=best_params['seasonal_years'], 
    trend_years=best_params['trend_years'], 
    week_weight=best_params['week_weight']
).fit(feat_train)
final_base.scales_ = base_model.scales_
final_base_pred = final_base.predict(feat_train)

X_train_final = feat_train[features]
y_train_rev_final = feat_train['Revenue'] - final_base_pred['Revenue']
y_train_cog_final = feat_train['COGS'] - final_base_pred['COGS']

X_train_final_aug, y_train_rev_final_aug, y_train_cog_final_aug = augment_data(X_train_final, y_train_rev_final, y_train_cog_final)

final_data_rev = lgb.Dataset(X_train_final_aug, label=y_train_rev_final_aug)
final_lgb_rev = lgb.train(lgb_params, final_data_rev, num_boost_round=1000)

final_data_cog = lgb.Dataset(X_train_final_aug, label=y_train_cog_final_aug)
final_lgb_cog = lgb.train(lgb_params, final_data_cog, num_boost_round=1000)

future_base = final_base.predict(feat_test)
future_rev = future_base['Revenue'] + final_lgb_rev.predict(feat_test[features])
future_cog = future_base['COGS'] + final_lgb_cog.predict(feat_test[features])

sub = pd.DataFrame({
    'Date': template['Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': future_rev.clip(lower=0).round(2),
    'COGS': future_cog.clip(lower=0).round(2)
})

out_path = OUT_DIR / 'submission_DA_Master_Breakthrough.csv'
sub.to_csv(out_path, index=False)
print(f"\n🎉 HOÀN TẤT! Đã xuất file thành công tại: {out_path}")